In [2]:
import sys, os
sys.path.append(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

import json

from bedrock_client import client, model_id_sonnet4
from bedrock_helpers import add_user_message, add_assistant_message, chat_haiku

In [3]:

def gernerate_dataset():
    prompt='''
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
'''
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat_haiku(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset = gernerate_dataset()
with open ("001_dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

In [7]:
def run_prompt(test_case):
    '''Merges the prompt and test case input,then returns the result'''
    prompt=f'''
Please solve the following task:
{test_case['task']}
'''
    messages = []
    add_user_message(messages, prompt)
    output = chat_haiku(messages)
    return output

In [13]:
def grade_by_model(test_case, output):
    '''Grades the results using the model itself.'''
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case['task']}
    Solution: {output}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and focused on the most important points.
    Example response:
    {{
        "strengths": "string[]",
        "weaknesses": "string[]",
        "reasoning": "string",
        "score": number
    }}

    """
   
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat_haiku(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [ ]:
import re
import ast

def validate_json(text):
    '''Validates if the text is a valid JSON object.'''
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0
    
def validate_python(text):
    '''Validates if the text is a valid Python code.'''
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0
    
def validate_regex(text):
    '''Validates if the text is a valid regular expression.'''
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, text_case):
    '''Grades the response based on the expected format.'''
    format = text_case['format']
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format: {format}")

In [14]:
def run_test_case(test_case):
    '''Calls run_prompt, then grades the result'''
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "score": score,
        "reasoning": reasoning,
        "test_case": test_case
    }

In [18]:
from statistics import mean
def run_eval(test_case):
    '''Loads the dataset and calls run_test_case with each case'''
    results =[]

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    average_score = mean([result["score"] for result in results])
    print('Average Score:', average_score)
    return results

In [19]:
with open("001_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average Score: 6.666666666666667


In [17]:
print(json.dumps(results, indent=4))

[
    {
        "output": "Here's a Python function that takes a list of AWS EC2 instance IDs and returns a list of their corresponding instance types:\n\n```python\nimport boto3\n\ndef get_instance_types(instance_ids):\n    \"\"\"\n    Retrieves the instance types for the given list of EC2 instance IDs.\n    \n    Args:\n        instance_ids (list): A list of EC2 instance IDs.\n        \n    Returns:\n        list: A list of instance types corresponding to the input instance IDs.\n    \"\"\"\n    ec2 = boto3.client('ec2')\n    \n    try:\n        response = ec2.describe_instances(InstanceIds=instance_ids)\n        instance_types = [instance['InstanceType'] for reservation in response['Reservations'] for instance in reservation['Instances']]\n        return instance_types\n    \n    except boto3.exceptions.ClientError as e:\n        print(f\"Error: {e}\")\n        return []\n```\n\nHere's how the function works:\n\n1. The function takes a list of EC2 instance IDs as input.\n2. It creat